# Node 1 — Inserter
Abre este notebook en una pestaña de JupyterHub.
Abre `node2_deleter.ipynb` en otra pestaña **al mismo tiempo**.
Ejecuta ambos con **Run All** — la tabla de sincronización hará que los commits ocurran a la vez.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────
CATALOG   = "players"
DATABASE  = "concurrency_tests"
TARGET    = f"{CATALOG}.{DATABASE}.concurrent_table"   # tabla que vas a modificar
SYNC      = f"{CATALOG}.{DATABASE}.sync_barrier"       # tabla de coordinación
NODE_ID   = "node1"                                    # identifica este notebook
OTHER_ID  = "node2"                                    # el otro notebook
# ──────────────────────────────────────────────────────────

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime
import time

def get_spark():
    builder = SparkSession.builder.appName(f"concurrency_{NODE_ID}")
    builder = builder.config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.367"
    )
    builder = builder.config(
        "spark.jars.excludes",
        "org.apache.hadoop:hadoop-client-runtime,org.apache.hadoop:hadoop-client-api"
    )
    builder = builder.config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    builder = builder.config("spark.sql.catalog.players",           "org.apache.iceberg.spark.SparkCatalog")
    builder = builder.config("spark.sql.catalog.players.type",      "rest")
    builder = builder.config("spark.sql.catalog.players.uri",       "http://172.16.58.11:32688")
    builder = builder.config("spark.sql.catalog.players.warehouse", "s3://warehouse/")
    builder = builder.config("spark.sql.catalog.players.io-impl",   "org.apache.iceberg.aws.s3.S3FileIO")
    builder = builder.config("spark.sql.catalog.players.s3.endpoint",            "http://172.16.58.11:31224")
    builder = builder.config("spark.sql.catalog.players.s3.region",              "us-east-1")
    builder = builder.config("spark.sql.catalog.players.s3.path-style-access",   "true")
    builder = builder.config("spark.sql.catalog.players.client.region",          "us-east-1")
    builder = builder.config("spark.hadoop.fs.s3a.endpoint",         "http://172.16.58.11:31224")
    builder = builder.config("spark.hadoop.fs.s3a.endpoint.region",  "us-east-1")
    builder = builder.config("spark.hadoop.fs.s3a.access.key",       "GK5f421d5f440758f74b0e0312")
    builder = builder.config("spark.hadoop.fs.s3a.secret.key",       "409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d")
    builder = builder.config("spark.hadoop.fs.s3a.path.style.access", "true")
    builder = builder.config("spark.hadoop.fs.s3a.impl",              "org.apache.hadoop.fs.s3a.S3AFileSystem")
    builder = builder.config("spark.sql.catalog.players.s3.checksum-algorithm",        "NONE")
    builder = builder.config("spark.sql.catalog.players.s3.streaming-upload-enabled",  "false")
    builder = builder.config("spark.sql.catalog.players.s3.chunked-encoding-enabled",  "false")
    builder = builder.config("spark.sql.catalog.players.s3.payload-signing-enabled",   "true")
    builder = builder.config("spark.sql.catalog.players.s3.http-client-type",          "urlconnection")
    builder = builder.config("spark.executor.extraJavaOptions", "-Daws.requestChecksumCalculation=when_required")
    builder = builder.config("spark.driver.extraJavaOptions",   "-Daws.requestChecksumCalculation=when_required")
    builder = builder.config("spark.sql.catalog.players.s3.access-key-id",     "GK5f421d5f440758f74b0e0312")
    builder = builder.config("spark.sql.catalog.players.s3.secret-access-key", "409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d")
    return builder.getOrCreate()

spark = get_spark()
print(f"✅ SparkSession lista — {NODE_ID}")

In [ ]:
# ── SETUP: crea las tablas si no existen ──────────────────
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{DATABASE}")

# Tabla principal de prueba
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET} (
        id     INT,
        source STRING,
        value  STRING,
        ts     TIMESTAMP
    ) USING iceberg
    TBLPROPERTIES (
        'write.merge.mode'         = 'merge-on-read',
        'commit.retry.num-retries' = '4',
        'commit.retry.min-wait-ms' = '100'
    )
""")

# Tabla de sincronización (barrera)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SYNC} (
        node_id  STRING,
        estado   STRING,
        ts       TIMESTAMP
    ) USING iceberg
""")

# Limpia la barrera de ejecuciones anteriores
spark.sql(f"DELETE FROM {SYNC} WHERE node_id = '{NODE_ID}'")

# Inserta datos semilla solo si la tabla está vacía
if spark.table(TARGET).count() == 0:
    schema = StructType([
        StructField("id",     IntegerType()),
        StructField("source", StringType()),
        StructField("value",  StringType()),
        StructField("ts",     TimestampType()),
    ])
    seed = [(i, "seed", f"val_{i}", datetime.now()) for i in range(1, 21)]
    spark.createDataFrame(seed, schema).writeTo(TARGET).append()
    print(f"Semilla insertada: 20 filas")

print(f"Filas actuales en {TARGET}: {spark.table(TARGET).count()}")
print("✅ Setup completado")

In [ ]:
# ── BARRERA DE SINCRONIZACIÓN ─────────────────────────────
# Escribe señal "ready" y espera a que el otro nodo también esté listo.
# Los dos commits ocurrirán al mismo tiempo real.

print(f"[{NODE_ID}] Señalizando 'ready'...")
spark.sql(f"INSERT INTO {SYNC} VALUES ('{NODE_ID}', 'ready', now())")

print(f"[{NODE_ID}] Esperando a {OTHER_ID}...")
timeout = 120  # segundos máximos de espera
t_start = time.time()

while True:
    listos = spark.sql(
        f"SELECT node_id FROM {SYNC} WHERE estado = 'ready'"
    ).rdd.map(lambda r: r[0]).collect()

    if OTHER_ID in listos:
        print(f"[{NODE_ID}] ✅ {OTHER_ID} está listo — ARRANCANDO")
        break

    if time.time() - t_start > timeout:
        raise TimeoutError(f"{OTHER_ID} no se unió en {timeout}s. ¿Está corriendo node2?")

    time.sleep(0.5)

In [ ]:
# ── OPERACIÓN PRINCIPAL ───────────────────────────────────
schema = StructType([
    StructField("id",     IntegerType()),
    StructField("source", StringType()),
    StructField("value",  StringType()),
    StructField("ts",     TimestampType()),
])

nuevas_filas = [(100 + i, NODE_ID, f"insert_{i}", datetime.now()) for i in range(10)]
df = spark.createDataFrame(nuevas_filas, schema)

t0 = time.perf_counter()
df.writeTo(TARGET).append()
duracion_ms = (time.perf_counter() - t0) * 1000

snap = spark.sql(
    f"SELECT snapshot_id, committed_at FROM {CATALOG}.{DATABASE}.{TARGET.split('.')[-1]}.snapshots "
    f"ORDER BY committed_at DESC LIMIT 1"
).collect()[0]

print(f"""\n{'='*50}
[{NODE_ID}] INSERT completado
  Filas insertadas : 10
  Duración         : {duracion_ms:.1f} ms
  Snapshot ID      : {snap['snapshot_id']}
  Committed at     : {snap['committed_at']}
{'='*50}""")

print(f"\nTotal filas en tabla ahora: {spark.table(TARGET).count()}")

In [ ]:
# ── HISTORIAL DE SNAPSHOTS ────────────────────────────────
# Útil para ver el orden real en que se registraron los commits
spark.sql(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM {CATALOG}.{DATABASE}.{TARGET.split('.')[-1]}.snapshots
    ORDER BY committed_at DESC
    LIMIT 10
""").show(truncate=80)